In [1]:
import numpy as np
import pandas as pd
import torch
import os

# Set device type
device = "mps" if torch.backends.mps.is_available() else "cpu"
# device = "cpu"

torch.set_default_dtype(torch.float32)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/brainsimulation/miniconda3/envs/eeg-torch/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/brainsimulation/miniconda3/envs/eeg-torch/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/brainsimulation/miniconda3/envs/eeg-torch/lib/python3.12/site-packages/ipykernel/

In [2]:
from sklearn.preprocessing import StandardScaler
import os

event_dict = {
    "HandStart": 1,
    "FirstDigitTouch": 2, 
    "BothStartLoadPhase": 3,
    "LiftOff": 4,
    "Replace": 5, 
    "BothReleased": 6
}

if os.path.exists("X_train.csv"):
    X_train = np.loadtxt("X_train.csv", delimiter=',')
    y_train = np.loadtxt("y_train.csv", delimiter=',')
else:
    X_train = []
    y_train = []

    for subject in range(1, 13):
        for series in range(1, 9):
            # Load dataset
            train_og = pd.read_csv(f"datadir/train/subj{subject}_series{series}_data.csv")
            train_events = pd.read_csv(f"datadir/train/subj{subject}_series{series}_events.csv")

            train = pd.merge(train_og, train_events, on="id")

            # Extract just channel recordings as X
            chnames = train_og.columns[1:]
            cur_X = train[chnames]

            for index in range(len(train_events)):
                cur_y = 0
                for event, i in event_dict.items():
                    if train_events.iloc[index][event] == 1:
                        cur_y = i
                y_train.append(cur_y)

            X_train.append(cur_X)

    X_train = np.concat(X_train, axis=0)

scaler = StandardScaler()
scaler.fit(X_train)

def scale(X):
    return scaler.transform(X)

In [3]:
scaler = StandardScaler()
scaler.fit(X_train)

def scale(X):
    return scaler.transform(X)

In [4]:
# np.savetxt("X_train.csv", X_train, delimiter=',')
# np.savetxt("y_train.csv", y_train, delimiter=',')

In [5]:
from torch.utils.data import Dataset, ConcatDataset
import torch

class EEGDataset(Dataset):

    event_dict = {
        "HandStart": 1,
        "FirstDigitTouch": 2, 
        "BothStartLoadPhase": 3,
        "LiftOff": 4,
        "Replace": 5, 
        "BothReleased": 6
    }

    def __init__(self, data_path, event_path, transform=None):
        self.data = pd.read_csv(data_path)
        self.events = pd.read_csv(event_path)

        self.transform = transform

        self.class_to_idx = self.event_dict
        self.classes = list(self.event_dict.keys())
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        y = 0
        for event, i in self.class_to_idx.items():
            if self.events.iloc[index][event] == 1:
                y = i
        # y = self.events.iloc[index][list(self.class_to_idx.keys())].to_numpy(dtype='float32')
        
        chnames = self.data.columns[1:]
        X = self.data.iloc[index][chnames].to_numpy(dtype='float32')

        if self.transform:
            X = self.transform(X.reshape(1, -1))
        
        X = torch.Tensor(X)

        return X, y

all_ds = [EEGDataset(f"datadir/train/subj{subject}_series{series}_data.csv", f"datadir/train/subj{subject}_series{series}_events.csv", scale) for series in range(1, 9) for subject in range(1, 13)]
ds = ConcatDataset(all_ds)

In [6]:
from torch.utils.data import DataLoader, random_split

val_size = int(0.2 * len(ds))
train_size = len(ds) - val_size
train_dataset, val_dataset = random_split(ds, [train_size, val_size])

train_batch_size = 32
val_batch_size = 128

train_loader = DataLoader(train_dataset, train_batch_size, True, num_workers=0)
val_loader = DataLoader(val_dataset, val_batch_size, False, num_workers=0)

In [7]:
from sklearn.utils.class_weight import compute_class_weight

# y_train_nozero = y_train[y_train != 0]

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print(class_weights)

[ 0.16378516  5.4901569  13.92987568  6.62534115  5.4901569   7.98183759
  5.4901569 ]


In [8]:
from torch import nn

class lnn(nn.Module):
    def __init__(self, input_size, n_classes):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x):
        return self.layers(x)

n_channels = ds[0][0].shape[1]
model = lnn(n_channels, 7)
model.to(device)

lnn(
  (layers): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.5, inplace=False)
    (9): Linear(in_features=64, out_features=7, bias=True)
  )
)

In [9]:
def _train_round(model, loader, optimizer, criterion, device, metric):
    model.train()

    train_loss = np.zeros(len(loader))
    y_pred_all, y_true_all = [], []

    for idx_batch, (batch_x, batch_y) in enumerate(loader):
        optimizer.zero_grad()

        batch_x = batch_x.to(device=device, dtype=torch.float32)
        batch_y = (batch_y).to(device=device, dtype=torch.int64)

        output = model(batch_x).squeeze()

        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        y_pred_all.append(torch.argmax(output, axis=1).cpu().numpy())
        y_true_all.append(batch_y.cpu().numpy())

        train_loss[idx_batch] = loss.item()
    
    y_pred = np.concatenate(y_pred_all)
    y_true = np.concatenate(y_true_all)
    perf = metric(y_true, y_pred)

    return np.mean(train_loss), perf

def _validate(model, loader, criterion, device, metric):
    model.eval()

    val_loss = np.zeros(len(loader))
    y_pred_all, y_true_all = [], []

    with torch.no_grad():
        for idx_batch, (batch_x, batch_y) in enumerate(loader):
            batch_x = batch_x.to(device=device, dtype=torch.float32)
            batch_y = (batch_y).to(device=device, dtype=torch.int64)
            
            output = model.forward(batch_x).squeeze()

            loss = criterion(output, batch_y)
            val_loss[idx_batch] = loss.item()
            
            y_pred_all.append(torch.argmax(output, axis=1).cpu().numpy())
            y_true_all.append(batch_y.cpu().numpy())
            
    y_pred = np.concatenate(y_pred_all)
    y_true = np.concatenate(y_true_all)
    perf = metric(y_true, y_pred)

    return np.mean(val_loss), perf

import copy

def train(model, loader_train, loader_valid, optimizer, criterion, n_epochs, patience, device, metric):

    best_valid_loss = np.inf
    best_model = copy.deepcopy(model)
    waiting = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        train_loss, train_perf = _train_round(model, loader_train, optimizer, criterion, device, metric=metric)
        valid_loss, valid_perf = _validate(model, loader_valid, criterion, device, metric=metric)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'valid_loss': valid_loss,
            'train_perf': train_perf,
            'valid_perf': valid_perf
        })

        print(f'{epoch} \t {train_loss:0.4f} \t {valid_loss:0.4f} 'f'\t {train_perf:0.4f} \t {valid_perf:0.4f}')

        # model saving
        if valid_loss < best_valid_loss:
            print(f'best val loss {best_valid_loss:.4f} -> {valid_loss:.4f}')
            best_valid_loss = valid_loss
            best_model = copy.deepcopy(model)
            waiting = 0
        else:
            waiting += 1

        # model early stopping
        if waiting >= patience:
            print(f'Stop training at epoch {epoch}')
            print(f'Best val loss : {best_valid_loss:.4f}')
            break

    return best_model, history

In [10]:
from torch.nn import CrossEntropyLoss
from sklearn.metrics import cohen_kappa_score
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=0)
criterion = CrossEntropyLoss(weight=torch.Tensor(class_weights))

In [11]:
n_epochs = 50
patience = 10

best_model, history = train(model, train_loader, val_loader, optimizer, criterion, n_epochs, patience, device, metric=cohen_kappa_score)

RuntimeError: Placeholder storage has not been allocated on MPS device!

In [ ]:
x = next(iter(train_loader))[0]
y = next(iter(train_loader))[1]

In [ ]:
x = next(iter(train_loader))[0]
x = model(x)
x.shape

torch.Size([32, 1, 6])